# Вебинары курса Рекомендательные системы


# Вебинар 1. Введение, примеры задач, бизнес- и ML-метрики 

-----

# 1. Введение и примеры задач

RecSys everywhere

##  Постановка задачи

*Задача рекомендаций* - предложить пользователю товары, которые ему "понравятся"

Компоненты:
    - Пользователи (users)
    - Товары (items)
    - Взаимодействия (interactions) - по ним понимаем, что товар "понравился"

---

## Пользователи VS Товары


**Кейс**: определить в каждой ситуации, что является пользователем, а что - товаром
    
    1. На сайте Кинопоиска порекомендовать интересные фильмы
    2. На сайте hh.ru подобрать лучшие вакансии для соискателя вакансий
    3. На сайте hh.ru найти похожие вакансии на ту, которую сейчас просматривает пользователь ("вам также может понравиться")
    3. На сайте hh.ru подобрать лучших кандидатов для работодателя
    
*Пример ответа в чате:*
    1. Пользователь (П) - фильм, Товар (T) - фильм
    2. П - работодатель, Т - соискатель вакансий

**Ответ:**
1. Кинопоиск, порекомендовать интересные фильмы: П — зритель (аккаунт на Кинопоиске), Т — фильм.
2. hh.ru, лучшие вакансии для соискателя: П — соискатель (его резюме), Т — вакансия.
3. hh.ru, похожие вакансии на просматриваемую («вам также может понравиться»): П — текущая вакансия, которую сейчас смотрят (item-to-item; можно расширить до «соискатель + текущая вакансия»), Т — другие вакансии.
4. hh.ru, лучшие кандидаты для работодателя: П — работодатель (точнее, его вакансия), Т — соискатель (резюме).

*В примере ответа из шаблона в пункте 1 «П — фильм», хотя пользователем там является зритель.*

---

## Взаимодействия --> "понравился товар"



---

## В чем отличие от классификации?

In [1]:
import pandas as pd
import numpy as np

In [2]:
pd.DataFrame([['Иван', 'Хлеб Бородинский', 1],
             ['Иван', 'Хлеб Белый', 0],
             ['Василий', 'Йогурт Epica', 1]], columns=['user', 'item', 'purchase_fact'])

,user,item,purchase_fact
0,Иван,Хлеб Бородинский,1
1,Иван,Хлеб Белый,0
2,Василий,Йогурт Epica,1


Ведь можно добавить фичи на user (средний чек, кол-во покупок в категориях и т.д), 
на item (цена, кол-во продаж в неделю и т.l), и решать задачу классификации. В чем отличие RecSys от классификации?

 - Много предсказаний для 1 user (extreme classification)
 - Гораздо больший объем данных: 100K users, 10K items --> 1B предсказаний
 - Большинство товаров user никогда не видел --> Не взаимодействовал --> 0 не значит "не понравился"
 - Нет явного таргета. Не понятно, что значит "(не) понравился"
 - Feedback loop
 - Иногда важен порядок рекомендаций

In [3]:
pd.DataFrame([['Иван', 'Хлеб Бородинский', 1],
              ['Иван', 'Хлеб Белый', 0],
              ['Иван', 'Йогурт Epica', "?"],
              ['Василий', 'Хлеб Бородинский', "?"],
              ['Василий', 'Хлеб Белый', "?"],
              ['Василий', 'Йогурт Epica', 1]], columns=['user', 'item', 'purchase_fact'])

,user,item,purchase_fact
0,Иван,Хлеб Бородинский,1
1,Иван,Хлеб Белый,0
2,Иван,Йогурт Epica,?
3,Василий,Хлеб Бородинский,?
4,Василий,Хлеб Белый,?
5,Василий,Йогурт Epica,1


----

## Основные типы задач:
    
**Рекомендация топ-К товаров**: Дан список товаров. Рекомендовать пользователю K товаров, которые ему понравятся
    - e-mail рассылки (М.Видео, Hoff, Пятерочка)
    - push-уведомления (Delivery Club, HeadHunter, Виктория)
    - Рекомендации в отдельном окне на сайте (vk, okko)

**Ранжирование товаров**: Дан список товаров. Нужно его отранжировать в порядке убывания интереса для пользователя
    - Ранжирование каталога товаров (Wildberries, Lamoda)
    - Ранжирование ленты (vk, Я.Дзен)
    - Ранжирование поисковой выдачи (Яндекс, Гугл)
    - Ранжирование "каруселей" товаров (Delivery Club, Я.Еда)
    
**Поиск похожих товаров**: Дан 1 товар. Нужно найти наиболее похожие на него товары
    - "Вам также может понравиться" (Ozon, Циан)
    - Похожим пользователям понравилось (vk)
    - Вы можете быть знакомы (vk, facebook)
    
**Рекомендация дополнителдьного товара**. Дан 1 товар. Найти товары, которые покупают вместе с этим товаром
    - С этим товаром часто покупают (Я.Маркет, Ozon, М.Видео)
    


---

# 2. ML-mетрики качества

*Хороши ли рекомендации, предлагаемые моделью?*  
Обычно считаются для каждого юзера, затем усредняются по юзерам

Представим, что рекомендовано всего 10 товаров
Куплено - 4 товара, некоторые не из рекомендованного списка

In [4]:
recommended_list = [143, 156, 1134, 991, 27, 1543, 3345, 533, 11, 43] #id товаров
bought_list = [521, 32, 143, 991]

In [5]:
len(recommended_list)

10

### 1. Hit rate

Hit rate = был ли хотя бы 1 релевантный товар среди рекомендованных

- Иногда применяется, когда продаются достаточно дрогие товары (например, бытовая техника) 

----
Hit rate = (был ли хотя бы 1 релевантный товар среди рекомендованных)   

Hit rate@k = (был ли хотя бы 1 релевантный товар среди топ-k рекомендованных)

In [6]:
def hit_rate(recommended_list, bought_list):

    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)

    flags = np.isin(bought_list, recommended_list)

    hit_rate = (flags.sum() > 0) * 1

    return hit_rate


def hit_rate_at_k(recommended_list, bought_list, k=5):

    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)[:k]   # берём только топ-k рекомендаций

    flags = np.isin(bought_list, recommended_list)

    hit_rate = (flags.sum() > 0) * 1

    return hit_rate

In [7]:
hit_rate(recommended_list, bought_list)

np.int64(1)

In [8]:
print('hit_rate@1 =', hit_rate_at_k(recommended_list, bought_list, k=1))  # 143 на 1-м месте -> 1
print('hit_rate@5 =', hit_rate_at_k(recommended_list, bought_list, k=5))
print('hit_rate@1 для списка без попаданий =', hit_rate_at_k([1, 2, 3], bought_list, k=1))

hit_rate@1 = 1
hit_rate@5 = 1
hit_rate@1 для списка без попаданий = 0


### 2. Precision

*Precision* - доля релевантных товаров среди рекомендованных = Какой % рекомендованных товаров  юзер купил

- Пожалуй, самая приближенная к бизнес-метрикам и самая популярная метрика

---

Precision= (# of recommended items that are relevant) / (# of recommended items)  

Precision@k = (# of recommended items @k that are relevant) / (# of recommended items @k)

Money Precision@k = (revenue of recommended items @k that are relevant) / (revenue of recommended items @k)  

**Note:** Обычно k в precision@k достаточно невелико (5-20) и определяется из бизнес-логики. Например, 5 товаров в e-mail рассылке, 20 ответов на первой странице google и т.д

Красная рыба - 400 руб  
Молоко - 60 руб  
Хлеб = 40 руб  
Гречка = 40 руб  
Шоколад = 90 руб  

------  
Варенье - 240 руб  
...  

**Case 1** 
рекомендовали - Красная рыба, Молоко, Хлеб, Гречка, Шоколад 
купили из рекомендованного 1 и последний товар


prices_reсommended = [400, 60, 40, 40 , 90]  
flags = [1, 0, 0, 0 , 1]  

$precison@5 = \frac{1 + 0 + 0 +0 + 1}{1+1+1+1+1} = 40\%$  
$money precision@5 = \frac{1*400 + 0*60 + ... + 1*90}{1*400 + 1*60 + ... + 1*90} = 77.7\%$  

  
**Case 2**   
рекомендовали - Молоко, Красная рыба,  Хлеб, Гречка, Шоколад 
купили из рекомендованного 1 и последний товар


prices_reсommended = [60, 400,  40, 40 , 90]  
flags = [1, 0,  0, 0 , 1]  

$precison@5 = \frac{0 + 1 + 0 +0 + 1}{1+1+1+1+1} = 40\%$  
$money precision@5 = \frac{0*400 + 1*60 + ... + 1*90}{1*400 + 1*60 + ... + 1*90} = 15.8\%$

In [9]:
def precision(recommended_list, bought_list):

    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)

    flags = np.isin(bought_list, recommended_list)

    precision = flags.sum() / len(recommended_list)

    return precision


def precision_at_k(recommended_list, bought_list, k=5):

    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)

    #assert len(bought_list) > len(recommended_list)
    bought_list = bought_list  # Тут нет [:k] !!
    recommended_list = recommended_list[:k]

    flags = np.isin(bought_list, recommended_list)

    precision = flags.sum() / len(recommended_list)


    return precision


def money_precision_at_k(recommended_list, bought_list, prices_recommended, k=5):

    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)[:k]
    prices_recommended = np.array(prices_recommended, dtype=float)[:k]

    # flags[i] = 1, если i-й рекомендованный товар куплен
    flags = np.isin(recommended_list, bought_list)

    # выручка релевантных рекомендаций / выручка всех рекомендаций (скалярное произведение, без цикла)
    precision = flags @ prices_recommended / prices_recommended.sum()

    return precision

In [10]:
precision(recommended_list, bought_list)

np.float64(0.2)

In [11]:
precision_at_k(recommended_list, bought_list, k=5)

np.float64(0.4)

In [12]:
precision_at_k(recommended_list, bought_list, k=3)

np.float64(0.3333333333333333)

In [13]:
# Проверим money precision@5 на кейсах из лекции
rec_case = ['Красная рыба', 'Молоко', 'Хлеб', 'Гречка', 'Шоколад']
bought_case = ['Красная рыба', 'Шоколад']
prices_case = [400, 60, 40, 40, 90]
print('Case 1: precision@5 = %.3f, money precision@5 = %.3f' % (
    precision_at_k(rec_case, bought_case, k=5), money_precision_at_k(rec_case, bought_case, prices_case, k=5)))

rec_case2 = ['Молоко', 'Красная рыба', 'Хлеб', 'Гречка', 'Шоколад']
bought_case2 = ['Молоко', 'Шоколад']
prices_case2 = [60, 400, 40, 40, 90]
print('Case 2: precision@5 = %.3f, money precision@5 = %.3f' % (
    precision_at_k(rec_case2, bought_case2, k=5), money_precision_at_k(rec_case2, bought_case2, prices_case2, k=5)))

Case 1: precision@5 = 0.400, money precision@5 = 0.778
Case 2: precision@5 = 0.400, money precision@5 = 0.238


*Примечание.* В Case 1 получаем 77.8%, как в лекции. В Case 2 по формуле выходит (60 + 90) / 630 ≈ 23.8%, а в тексте лекции стоит 15.8% — похоже на опечатку в слайде: при тех же флагах и ценах числитель равен 150.

### 3. Recall

*Recall* - доля рекомендованных товаров среди релевантных = Какой % купленных товаров был среди рекомендованных

- Обычно используется для моделей пре-фильтрации товаров (убрать те товары, которые точно не будем рекомендовать)

---

Recall= (# of recommended items that are relevant) / (# of relevant items)  

Recall@k = (# of recommended items @k that are relevant) / (# of relevant items)

Money Recall@k = (revenue of recommended items @k that are relevant) / (revenue of relevant items)  

    
  
**Note:** в recall@k число k обычно достаточно большое (50-200), больше чем покупок у среднестатистического юзера

In [14]:
def recall(recommended_list, bought_list):

    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)

    flags = np.isin(bought_list, recommended_list)

    recall = flags.sum() / len(bought_list)

    return recall


def recall_at_k(recommended_list, bought_list, k=5):

    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)[:k]

    flags = np.isin(bought_list, recommended_list)

    recall = flags.sum() / len(bought_list)

    return recall


def money_recall_at_k(recommended_list, bought_list, prices_recommended, prices_bought, k=5):

    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)[:k]
    prices_recommended = np.array(prices_recommended, dtype=float)[:k]
    prices_bought = np.array(prices_bought, dtype=float)

    flags = np.isin(recommended_list, bought_list)

    # выручка релевантных товаров из топ-k / выручка всех купленных (релевантных) товаров
    recall = flags @ prices_recommended / prices_bought.sum()

    return recall

In [15]:
recall(recommended_list, bought_list)

np.float64(0.5)

In [16]:
prices_recommended = [400, 60, 40, 40, 90, 240, 100, 55, 30, 70]   # цены для recommended_list (условные)
prices_bought = [150, 35, 400, 40]                                 # цены для bought_list: 521, 32, 143, 991

print('recall@5 =', recall_at_k(recommended_list, bought_list, k=5))
print('recall@10 =', recall_at_k(recommended_list, bought_list, k=10))
print('money recall@5 = %.3f' % money_recall_at_k(recommended_list, bought_list, prices_recommended, prices_bought, k=5))
print('money precision@5 = %.3f' % money_precision_at_k(recommended_list, bought_list, prices_recommended, k=5))

recall@5 = 0.5
recall@10 = 0.5
money recall@5 = 0.704
money precision@5 = 0.698


---

# Метрики ранжирования

Если важен порядок рекомендаций. Подробнее можно почитать [здесь](https://habr.com/ru/company/econtenta/blog/303458/). Формулы в статье могут несколько отличаться от формул в лекции 

## AP@k
AP@k - average precision at k

$$AP@k = \frac{1}{r} \sum{[recommended_{relevant_i}] * precision@k}$$

- r - кол-во релевантный среди рекомендованных
- Суммируем по всем релевантным товарам
- Зависит от порядка реокмендаций

In [17]:
def ap_k(recommended_list, bought_list, k=5):

    bought_list = np.array(bought_list)
    recommended_list = np.array(recommended_list)[:k]   # исправлено: смотрим только на топ-k

    flags = np.isin(recommended_list, bought_list)

    if sum(flags) == 0:
        return 0

    sum_ = 0
    for i in range(0, len(recommended_list)):            # исправлено: в шаблоне было range(0, k-1), k-я позиция терялась
        if flags[i] == True:
            p_k = precision_at_k(recommended_list, bought_list, k=i+1)
            sum_ += p_k

    result = sum_ / sum(flags)

    return result

In [18]:
ap_k(recommended_list, bought_list, k=5)

np.float64(0.75)

### MAP@k

MAP@k (Mean Average Precision@k)  
Среднее AP@k по всем юзерам
- Показывает средневзвешенную точность рекомендаций

$$MAP@k = \frac{1}{|U|} \sum_u{AP_k}$$
  
|U| - кол-во юзеров

In [19]:
recommended_list = [[143, 156, 1134, 991, 27, 1543, 3345, 533, 11, 43],
                    [146, 156, 1134, 991, 27, 1543, 3345, 533, 11, 43],] #id товаров
bought_list = [[521, 32, 143, 991], [146, 29]]

In [20]:
def map_k(recommended_list, bought_list, k=5, u=1):
    """MAP@k: среднее AP@k по пользователям.
    recommended_list, bought_list — списки списков (по одному на юзера).
    u — число пользователей |U|; если передано значение, не совпадающее с числом списков, берём реальное число юзеров."""

    if u != len(bought_list):
        u = len(bought_list)

    apk_values = [ap_k(rec, bought, k=k) for rec, bought in zip(recommended_list, bought_list)]
    result = np.sum(apk_values) / u

    return result

In [21]:
print('AP@5 по юзерам:', [ap_k(r, b, k=5) for r, b in zip(recommended_list, bought_list)])
print('MAP@5 =', map_k(recommended_list, bought_list, k=5))

AP@5 по юзерам: [np.float64(0.75), np.float64(1.0)]
MAP@5 = 0.875


### AUC@k
AUC для первых k наблюдений  
- Можно посчитать как sklern.metrics.roc_auc_score для топ-k предсказаний
- Показывает долю верно отранжированных товаров

In [22]:
# AUC@k: доля верно упорядоченных пар (релевантный, нерелевантный) среди первых k рекомендаций
from sklearn.metrics import roc_auc_score

def auc_at_k(recommended_list, bought_list, k=5):
    recommended_list = np.array(recommended_list)[:k]
    flags = np.isin(recommended_list, bought_list).astype(int)
    if flags.sum() in (0, len(flags)):      # AUC не определён, если в топ-k только один класс
        return np.nan
    scores = np.arange(len(recommended_list), 0, -1)   # чем выше позиция, тем больше скор
    return roc_auc_score(flags, scores)

auc_at_k(recommended_list[0], bought_list[0], k=5)

0.6666666666666667

### NDCG@k
Normalized discounted cumulative gain

$DCG = \frac{1}{|r|} \sum_u{\frac{[bought fact]}{discount(i)}}$  

$discount(i) = 1$ if $i <= 2$,   
$discount(i) = log_2(i)$ if $i > 2$


(!) Считаем для первых k рекоммендаций   
(!) - существуют вариации с другими $discount(i)$  
i - ранк рекомендованного товара  
|r| - кол-во рекомендованных товаров 

$$NDCG = \frac{DCG}{ideal DCG}$$


https://pytorch.org/docs/stable/index.html

In [23]:
import torch
torch.manual_seed(42)   # фиксируем случайность, чтобы числа ниже воспроизводились
N = 5
ys_true = torch.randint(0, 5, (N, ))
ys_pred = torch.rand(N)

In [24]:
ys_true

tensor([2, 2, 1, 4, 1])

In [25]:
ys_pred

tensor([0.6009, 0.2566, 0.7936, 0.9408, 0.1332])

In [26]:
# итоговый результат модели - отсортированный ys_pred
ys_pred

tensor([0.6009, 0.2566, 0.7936, 0.9408, 0.1332])

In [27]:
ys_pred_np = ys_pred.numpy()   # numpy-копия; сам ys_pred оставляем тензором — он нужен в torch.sort ниже

In [28]:
ys_pred_np

array([0.60089535, 0.25657248, 0.7936413 , 0.94077146, 0.13318592],
      dtype=float32)

In [29]:
def compute_gain(y_value: float, gain_scheme: str) -> float:
    if gain_scheme == "exp2":
        gain = 2 ** y_value - 1
    elif gain_scheme == "const":
        gain = y_value
    else:
        raise ValueError(f"{gain_scheme} method not supported, only exp2 and const.")
    return float(gain)

In [30]:
import math

def dcg(ys_true: torch.Tensor, ys_pred: torch.Tensor, gain_scheme: str) -> float:
    _, argsort = torch.sort(ys_pred, descending=True, dim=0)
    ys_true_sorted = ys_true[argsort]
    ret = 0
    for idx, cur_y in enumerate(ys_true_sorted, 1):
        gain = compute_gain(cur_y, gain_scheme)
        ret += gain / math.log2(idx + 1)   # дисконт log2(i + 1): 1-я позиция без штрафа
    return ret

In [31]:
_, argsort = torch.sort(ys_pred, descending=True, dim=0)

In [32]:
ys_pred

tensor([0.6009, 0.2566, 0.7936, 0.9408, 0.1332])

In [33]:
argsort

tensor([3, 2, 0, 1, 4])

In [34]:
ys_true_sorted = ys_true[argsort]

In [35]:
for idx, cur_y in enumerate(ys_true_sorted, 1):
    print(idx, cur_y)

1 tensor(4)
2 tensor(1)
3 tensor(2)
4 tensor(2)
5 tensor(1)


In [36]:
def ndcg(ys_true: torch.Tensor, ys_pred: torch.Tensor, gain_scheme: str = 'const') -> float:
    pred_dcg = dcg(ys_true, ys_pred, gain_scheme)
    # идеальный DCG — если отсортировать объекты по истинной релевантности
    ideal_dcg = dcg(ys_true, ys_true, gain_scheme)

    if ideal_dcg == 0:
        return 0.0
    ndcg = pred_dcg / ideal_dcg
    return ndcg

In [37]:
print('ys_true =', ys_true, ' ys_pred =', ys_pred)
ys_pred_t = ys_pred
for scheme in ['const', 'exp2']:
    print(f'{scheme}: DCG = {dcg(ys_true, ys_pred_t, scheme):.4f}, '
          f'NDCG = {ndcg(ys_true, ys_pred_t, scheme):.4f}, '
          f'NDCG(идеальное ранжирование) = {ndcg(ys_true, ys_true.float(), scheme):.4f}')

ys_true = tensor([2, 2, 1, 4, 1])  ys_pred = tensor([0.6009, 0.2566, 0.7936, 0.9408, 0.1332])
const: DCG = 6.8791, NDCG = 0.9717, NDCG(идеальное ранжирование) = 1.0000
exp2: DCG = 18.8098, NDCG = 0.9792, NDCG(идеальное ранжирование) = 1.0000


In [38]:
# Вариант NDCG@k из лекции (бинарная релевантность «купил / не купил», discount(i) = 1 при i <= 2, иначе log2(i))
def ndcg_at_k(recommended_list, bought_list, k=5):
    recommended_list = np.array(recommended_list)[:k]
    flags = np.isin(recommended_list, bought_list).astype(float)
    ranks = np.arange(1, len(recommended_list) + 1)
    discount = np.where(ranks <= 2, 1.0, np.log2(ranks))
    dcg_ = (flags / discount).sum() / len(recommended_list)
    ideal_flags = np.zeros(len(recommended_list))
    ideal_flags[:min(len(bought_list), len(recommended_list))] = 1   # идеал: все купленные товары в начале списка
    idcg_ = (ideal_flags / discount).sum() / len(recommended_list)
    return dcg_ / idcg_ if idcg_ > 0 else 0.0

print('NDCG@5 (вариант из лекции) =', round(ndcg_at_k(recommended_list[0], bought_list[0], k=5), 4))

NDCG@5 (вариант из лекции) = 0.4791


### MRR@k
Mean Reciprocal Rank

- Считаем для первых k рекоммендаций
- Найти ранк первого релевантного предсказания $k_u$
- Посчитать reciprocal rank = $\frac{1}{k_u}$

$$MRR = mean(\frac{1}{k_u})$$

[MRR](https://drive.google.com/file/d/1uMFKqWULFmehS0pJtvXKRygjYxQOPise/view?usp=sharing)

In [39]:
def reciprocal_rank(recommended_list, bought_list, k=5):
    """RR@k для одного юзера: 1 / ранг первого релевантного товара среди первых k рекомендаций (0, если попаданий нет)."""
    recommended_list = np.array(recommended_list)[:k]
    flags = np.isin(recommended_list, bought_list)
    if not flags.any():
        return 0.0
    first_rank = np.argmax(flags) + 1   # argmax возвращает индекс первого True
    result = 1 / first_rank
    return result


def mrr_at_k(recommended_list, bought_list, k=5):
    """MRR@k: среднее RR@k по пользователям (списки списков)."""
    return np.mean([reciprocal_rank(r, b, k=k) for r, b in zip(recommended_list, bought_list)])

In [40]:
print('RR@5 юзера 1 =', reciprocal_rank(recommended_list[0], bought_list[0], k=5))   # первое попадание — 1-я позиция
print('RR@5 для [27, 991, 143] =', reciprocal_rank([27, 991, 143], bought_list[0], k=5))  # первое попадание — 2-я позиция
print('MRR@5 =', mrr_at_k(recommended_list, bought_list, k=5))

RR@5 юзера 1 = 1.0
RR@5 для [27, 991, 143] = 0.5
MRR@5 = 1.0


# 3. Связь бизнес-метрик, ML-метрик и функции потерь(loss)

- **loss** - то, что оптимизирует модель (RMSE - Root Mean Squared Error)
- **ML-метрика** - то, как мы измеряем качество на test (money precision@5)
- **Бизнес-метрика** - то, что хочет оптимизировать бизнес (выручка)

В идеале loss = ML-метрика = бизнес-метрика, но это возмонжо в очень редких случаях

Надеемся, что:  
    *Оптимизация loss --> рост ML-метрик --> рост бизнес-метрик*

### Пример: рекомендательная система товаров на сайте  
      
На сайте есть баннер, на котором мы можем разместить 5 товаров. Надо порекомендовать эти 5 товаров персонально каждому юзеру

*Шаг 1: Определим бизнес-метрику*  
Бизнес хочет максимизировать выручку --> бизнес-метрика - **Выручка**  

*Шаг 2: Разложим ее на составляющие*  
Выручка =   
       Средний чек * кол-во покупок =   
       Средний чек * (число юзеров * конверсия из захода на сайт в заказ) =
       Число юзеров * (Средний чек * конверсия из захода на сайт в заказ)
       
Рекомендательная система влияет только на (Средний чек * конверсия из захода на сайт в заказ).   
Хорошим приближением этого является **money precision@5** - ML-метрика

*Шаг 3: loss*  
Это существенно сложнее. Персонализированные ML-модели не умеют напрямую оптимизировать конверсию. Стандартный loss - **RMSE**. Можно для начала попробовать его. 

P.S. Если вы хотите приблизить RMSE к money precision@5, то можно посчитать weighted RMSE, где вес каждого наблюдения = стоимость товара.   
P.S.S. Про другие виды loss будет рассказано в курсе